In [2]:

import re, random, sys
random.seed(42)

# Carica carte da mazzi.ts
with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT_STATE = {
    'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,
    'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
    'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
    'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':3,'stabilita_economica_russia':5,'stabilita_russia':5,
    'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
    'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5,
}
LIMITS = {'nucleare':(1,15),'sanzioni':(1,20),'opinione':(-10,10),'defcon':(1,10),'veto_onu_russia':(0,3)}
def clamp(key,val):
    lo,hi = LIMITS.get(key,(1,12))
    return max(lo,min(hi,val))

# Carica tutte le carte con effetti
ALL_T = list(DEFAULT_STATE.keys())
card_re = re.compile(r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)',\s*op_points:(\d+),\s*deck_type:'([^']+)'.*?effects:\{([^}]+)\}", re.DOTALL)

ALL_CARDS = {}
for m in card_re.finditer(src):
    cid,cname,faz,ctype,op,deck,eff_str = m.groups()
    if deck in ('speciale','speciale_locked'): continue  # solo carte base
    effects = {}
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            try: effects[t] = eval(f"lambda v:{fn.group(1).strip().rstrip(',')}")
            except: pass
    ALL_CARDS[cid] = {'id':cid,'name':cname,'faction':faz,'type':ctype,'op':int(op),'effects':effects}

print(f"Carte caricate: {len(ALL_CARDS)}")
for f in ['Iran','Coalizione','Russia','Cina','Europa']:
    n = sum(1 for c in ALL_CARDS.values() if c['faction']==f)
    print(f"  {f}: {n} carte")


Carte caricate: 332
  Iran: 64 carte
  Coalizione: 64 carte
  Russia: 58 carte
  Cina: 58 carte
  Europa: 58 carte


In [5]:

import re, random
random.seed(42)

with open('/workspace/linea_rossa_tracker/src/data/mazzi.ts') as f:
    src = f.read()

DEFAULT_STATE = {
    'nucleare':5,'sanzioni':10,'opinione':0,'defcon':7,
    'risorse_iran':5,'forze_militari_iran':5,'tecnologia_nucleare_iran':5,'stabilita_iran':5,
    'risorse_coalizione':5,'influenza_diplomatica_coalizione':5,'tecnologia_avanzata_coalizione':5,'supporto_pubblico_coalizione':5,'stabilita_coalizione':5,
    'risorse_russia':5,'influenza_militare_russia':5,'veto_onu_russia':3,'stabilita_economica_russia':5,'stabilita_russia':5,
    'risorse_cina':5,'influenza_commerciale_cina':5,'cyber_warfare_cina':5,'stabilita_rotte_cina':5,'stabilita_cina':5,
    'risorse_europa':5,'influenza_diplomatica_europa':5,'aiuti_umanitari_europa':5,'coesione_ue_europa':5,'stabilita_europa':5,
}
LIMITS = {'nucleare':(1,15),'sanzioni':(1,20),'opinione':(-10,10),'defcon':(1,10),'veto_onu_russia':(0,3)}

def clamp(key, val):
    lo, hi = LIMITS.get(key, (1, 12))
    return max(lo, min(hi, val))

ALL_T = list(DEFAULT_STATE.keys())
card_re = re.compile(
    r"card_id:'([^']+)',\s*card_name:'([^']+)',\s*faction:'([^']+)',\s*card_type:'([^']+)',\s*op_points:(\d+),\s*deck_type:'([^']+)'.*?effects:\{([^}]+)\}",
    re.DOTALL
)

ALL_CARDS = {}
for m in card_re.finditer(src):
    cid, cname, faz, ctype, op, deck, eff_str = m.groups()
    if deck in ('speciale', 'speciale_locked'):
        continue
    effects = {}
    for t in ALL_T:
        fn = re.search(rf'{t}\s*:\s*\(v[^)]*\)\s*=>\s*([^,}}]+)', eff_str)
        if fn:
            try:
                effects[t] = eval(f"lambda v:{fn.group(1).strip().rstrip(',')}")
            except:
                pass
    ALL_CARDS[cid] = {'id': cid, 'name': cname, 'faction': faz, 'type': ctype, 'op': int(op), 'effects': effects}

FAZIONI = ['Iran', 'Coalizione', 'Russia', 'Cina', 'Europa']
ORDER   = ['Iran', 'Coalizione', 'Russia', 'Cina', 'Europa']

def deal_hand(faction, n=5):
    pool = [c for c in ALL_CARDS.values() if c['faction'] == faction]
    random.shuffle(pool)
    return pool[:n]

hands = {f: deal_hand(f) for f in FAZIONI}
deck  = {f: [c for c in ALL_CARDS.values() if c['faction'] == f][5:] for f in FAZIONI}
for f in FAZIONI:
    random.shuffle(deck[f])

def draw_card(faction):
    if deck[faction]:
        return deck[faction].pop(0)
    return None

FACTION_WEIGHTS = {
    'Iran':       {'nucleare':3,'sanzioni':-3,'opinione':-2,'defcon':-1,'risorse':2,'stabilita':2,'risorse_iran':2,'forze_militari_iran':2,'tecnologia_nucleare_iran':3,'stabilita_iran':2},
    'Coalizione': {'nucleare':-3,'sanzioni':3,'opinione':2,'defcon':1,'risorse':2,'stabilita':2,'risorse_coalizione':2,'tecnologia_avanzata_coalizione':2,'tecnologia_nucleare_iran':-3,'forze_militari_iran':-2,'stabilita_iran':-2},
    'Russia':     {'nucleare':1,'sanzioni':-2,'opinione':-1,'defcon':-1,'risorse':2,'stabilita':2,'risorse_russia':2,'influenza_militare_russia':2,'veto_onu_russia':2,'forze_militari_iran':1},
    'Cina':       {'nucleare':1,'sanzioni':-2,'opinione':-1,'defcon':-1,'risorse':2,'stabilita':2,'risorse_cina':2,'influenza_commerciale_cina':2,'stabilita_rotte_cina':2},
    'Europa':     {'nucleare':-2,'sanzioni':2,'opinione':2,'defcon':2,'risorse':2,'stabilita':2,'risorse_europa':2,'influenza_diplomatica_europa':2,'aiuti_umanitari_europa':2,'coesione_ue_europa':2},
}

def score_card(card, faction, state):
    w = FACTION_WEIGHTS.get(faction, {})
    s = 0
    for t, fn in card['effects'].items():
        try:
            delta = fn(state.get(t, 5))
            s += delta * w.get(t, 0)
        except:
            pass
    return s

def apply_card(state, card, faction):
    new = dict(state)
    deltas = {}
    for t, fn in card['effects'].items():
        if t in new:
            try:
                d = fn(new[t])
                new[t] = clamp(t, new[t] + d)
                if d != 0:
                    deltas[t] = d
            except:
                pass
    if faction != 'Russia' and new.get('sanzioni', 0) > state.get('sanzioni', 0):
        if state.get('veto_onu_russia', 0) > 0:
            new['sanzioni'] = state['sanzioni']
            new['veto_onu_russia'] = state['veto_onu_russia'] - 1
            deltas['[VETO]'] = -1
    return new, deltas

def check_win(state, turn):
    if state['nucleare'] >= 15:
        return 'Iran', 'Breakout Nucleare ☢️'
    if state['defcon'] <= 1:
        return 'Tutti perdono', 'Guerra Nucleare 💥'
    if state['sanzioni'] >= 20:
        return 'Coalizione', 'Sanzioni Totali 🔒'
    if turn >= 20:
        iran_score = state['nucleare']*3 + (20-state['sanzioni'])*2 + state['risorse_iran'] + state['stabilita_iran']
        coal_score = state['sanzioni']*2 + (15-state['nucleare'])*3 + state['risorse_coalizione'] + state['supporto_pubblico_coalizione']
        if iran_score > coal_score:
            return 'Iran', f'Punteggio finale — Iran {iran_score} vs Coalizione {coal_score}'
        else:
            return 'Coalizione', f'Punteggio finale — Iran {iran_score} vs Coalizione {coal_score}'
    return None, None

# ── SIMULAZIONE ──────────────────────────────────────────────────────────────
state = dict(DEFAULT_STATE)
turn  = 0
winner = None
win_reason = None

W = 68
print("=" * W)
print("  LINEA ROSSA — SIMULAZIONE PARTITA  (seed=42)")
print("=" * W)
print(f"{'Turno':<7} {'Fazione':<13} {'Carta':<32} {'Effetti chiave'}")
print("-" * W)

while turn < 20:
    turn += 1
    for faction in ORDER:
        if not hands[faction]:
            continue

        best = max(hands[faction], key=lambda c: score_card(c, faction, state))
        hands[faction].remove(best)

        new_card = draw_card(faction)
        if new_card:
            hands[faction].append(new_card)

        state, deltas = apply_card(state, best, faction)

        eff_parts = []
        for k, v in list(deltas.items())[:4]:
            if isinstance(v, float) and v == int(v):
                v = int(v)
            sign = '+' if (isinstance(v, (int, float)) and v > 0) else ''
            eff_parts.append(f"{k}{sign}{v}")
        eff_str = '  '.join(eff_parts)

        print(f"T{turn:<6} {faction:<13} {best['name'][:30]:<32} {eff_str}")

        # controlla vittoria dopo ogni giocata (non solo a fine round)
        end_of_round = (faction == ORDER[-1])
        winner, win_reason = check_win(state, turn if end_of_round else 0)
        if winner:
            break
    if winner:
        break

print()
print("=" * W)
print(f"  FINE PARTITA — Turno {turn}")
print("=" * W)
print()
print(f"  🏆  VINCITORE : {winner or 'Nessuno'}")
if win_reason:
    print(f"  📋  Motivo    : {win_reason}")
print()
print("  TRACCIATI GLOBALI ─────────────────────────────────")
print(f"  ☢️   Nucleare  : {state['nucleare']:>2} / 15")
print(f"  🔒  Sanzioni  : {state['sanzioni']:>2} / 20")
print(f"  📣  Opinione  : {state['opinione']:>+3}   (-10 / +10)")
print(f"  🚨  DEFCON    : {state['defcon']:>2} / 10")
print(f"  🏛️   Veto Rus  : {state['veto_onu_russia']:>2} / 3")
print()
print("  PLANCE FAZIONE ─────────────────────────────────────")
print(f"  IRAN      Risorse:{state['risorse_iran']}  Forze:{state['forze_militari_iran']}  Tecn.Nuc:{state['tecnologia_nucleare_iran']}  Stab:{state['stabilita_iran']}")
print(f"  COAL.     Risorse:{state['risorse_coalizione']}  InfDip:{state['influenza_diplomatica_coalizione']}  TecAv:{state['tecnologia_avanzata_coalizione']}  Supp:{state['supporto_pubblico_coalizione']}")
print(f"  RUSSIA    Risorse:{state['risorse_russia']}  InfMil:{state['influenza_militare_russia']}  StabEc:{state['stabilita_economica_russia']}")
print(f"  CINA      Risorse:{state['risorse_cina']}  InfCom:{state['influenza_commerciale_cina']}  Cyber:{state['cyber_warfare_cina']}")
print(f"  EUROPA    Risorse:{state['risorse_europa']}  InfDip:{state['influenza_diplomatica_europa']}  Coesione:{state['coesione_ue_europa']}")


  LINEA ROSSA — SIMULAZIONE PARTITA  (seed=42)
Turno   Fazione       Carta                            Effetti chiave
--------------------------------------------------------------------
T1      Iran          Rappresaglia Missilistica        nucleare+6  defcon-2  forze_militari_iran+1  stabilita_coalizione-1
T1      Coalizione    Strike Chirurgico                sanzioni+2  defcon-2  risorse_iran-1  forze_militari_iran-1
T1      Russia        Mediazione Pechino               sanzioni-1  influenza_diplomatica_coalizione-1  veto_onu_russia+1  stabilita_russia+1
T1      Cina          Mediazione Diplomatica Cina      sanzioni-1  opinione+1  risorse_cina+1  influenza_commerciale_cina+1
T1      Europa        JCPOA 3.0 Proposta UE            nucleare-2  sanzioni+3  opinione+2  defcon+2
T2      Iran          Centrifughe Avanzate             forze_militari_iran+1  tecnologia_nucleare_iran+1  stabilita_coalizione-1
T2      Coalizione    Aiuti Militari Israele           sanzioni+1  risorse_iran-1 